In [1]:
library(pacman)

p_load(shiny,
bslib,
leaflet,
dplyr,
lubridate,
ggplot2)



In [2]:
# ==============================================================================
# 1. LOAD & PREPROCESS
# ==============================================================================
script_path <- tryCatch(
  normalizePath(sys.frames()[[1]]$ofile, mustWork=FALSE),
  error=function(e) NULL
)
if (is.null(script_path) || is.na(script_path)) {
  args <- commandArgs(trailingOnly=FALSE)
  fa   <- grep("--file=", args, value=TRUE)
  if (length(fa)) script_path <- normalizePath(sub("--file=","",fa[1]), mustWork=FALSE)
}
script_dir <- if (!is.null(script_path) && !is.na(script_path)) dirname(script_path) else getwd()
data_dir   <- file.path(script_dir, "data")
if (!dir.exists(data_dir)) data_dir <- getwd()

csv_files <- list.files(data_dir, pattern="\\.csv$", full.names=TRUE)
if (!length(csv_files)) stop("No CSV files in: ", data_dir)

message("Loading ", length(csv_files), " CSVs...")
raw <- bind_rows(lapply(csv_files, read.csv, stringsAsFactors=FALSE))

cn <- names(raw)
cn[cn=="location.long"]               <- "lon"
cn[cn=="location.lat"]                <- "lat"
cn[cn=="individual.local.identifier"] <- "bird_id"
cn[cn=="lat.lower"]                   <- "lat_lower"
cn[cn=="lat.upper"]                   <- "lat_upper"
cn[cn=="long.lower"]                  <- "lon_lower"
cn[cn=="long.upper"]                  <- "lon_upper"
cn[cn=="study.name"]                  <- "study_name"
cn[cn=="behavioural.classification"]  <- "behaviour"
names(raw) <- cn

set.seed(42)
bird_ids_all <- unique(raw$bird_id[!is.na(raw$bird_id) & raw$bird_id != ""])
jitter_tbl <- tibble(
  bird_id = bird_ids_all,
  jit_lat = runif(length(bird_ids_all), -0.12, 0.12),
  jit_lon = runif(length(bird_ids_all), -0.12, 0.12)
)

swift_clean <- raw %>%
  filter(!is.na(lon), !is.na(lat), lon != "", lat != "") %>%
  mutate(
    timestamp = as.Date(ymd_hms(timestamp)),
    lat_mid   = ifelse(!is.na(lat_lower) & !is.na(lat_upper),
                       (as.numeric(lat_lower) + as.numeric(lat_upper)) / 2,
                       as.numeric(lat)),
    lon_mid   = ifelse(!is.na(lon_lower) & !is.na(lon_upper),
                       (as.numeric(lon_lower) + as.numeric(lon_upper)) / 2,
                       as.numeric(lon)),
    colony = trimws(gsub(" - Long term study.*", "", study_name))
  ) %>%
  left_join(jitter_tbl, by = "bird_id") %>%
  mutate(
    map_lat  = lat_mid + jit_lat,
    map_lon  = lon_mid + jit_lon,
    plot_lat = lat_mid
  ) %>%
  filter(!is.na(map_lat), map_lon >= -30, map_lon <= 65, map_lat >= -20, map_lat <= 75) %>%
  arrange(bird_id, timestamp)

unique_colonies <- sort(unique(swift_clean$colony))
unique_birds    <- sort(unique(swift_clean$bird_id))
colony_palette  <- colorFactor(palette = "Paired", domain = unique_colonies)
colony_colors   <- setNames(colony_palette(unique_colonies), unique_colonies)

message("Clean rows: ", nrow(swift_clean), " | Birds: ", length(unique_birds))

# ---- Pre-interpolate (one-time at startup) ----------------------------------
interpolate_bird <- function(df) {
  df <- arrange(df, timestamp)
  if (nrow(df) < 2) return(df %>%
    select(bird_id, colony, timestamp, map_lat, map_lon, plot_lat) %>%
    mutate(interp = FALSE))
  t_num  <- as.numeric(df$timestamp)
  t_out  <- as.numeric(seq(min(df$timestamp), max(df$timestamp), by = "day"))
  lon_i  <- approx(t_num, df$map_lon,  xout = t_out, rule = 2)$y
  lat_i  <- approx(t_num, df$map_lat,  xout = t_out, rule = 2)$y
  plat_i <- approx(t_num, df$plot_lat, xout = t_out, rule = 2)$y
  for (g in which(diff(t_num) > 20)) {
    m <- t_out > t_num[g] & t_out < t_num[g + 1]
    lon_i[m] <- NA; lat_i[m] <- NA; plat_i[m] <- NA
  }
  tibble(
    bird_id   = df$bird_id[1],
    colony    = df$colony[1],
    timestamp = as.Date(t_out, origin = "1970-01-01"),
    map_lat   = lat_i, map_lon = lon_i, plot_lat = plat_i,
    interp    = TRUE
  )
}

message("Pre-computing interpolated tracks...")
swift_interp <- swift_clean %>%
  group_by(bird_id) %>%
  group_modify(~interpolate_bird(.x)) %>%
  ungroup()
message("Ready. Interpolated rows: ", nrow(swift_interp))

SPERM_N <- 5

# ---- Batched polyline builders (NA-separator trick) -------------------------
make_colony_lines <- function(df) {
  birds <- unique(df$bird_id)
  parts <- lapply(birds, function(b) {
    bd <- df[df$bird_id == b, ]
    if (nrow(bd) < 2) return(NULL)
    rbind(bd[, c("map_lon","map_lat")],
          data.frame(map_lon = NA_real_, map_lat = NA_real_))
  })
  parts <- Filter(Negate(is.null), parts)
  if (!length(parts)) return(NULL)
  do.call(rbind, parts)
}

make_sperm_lines <- function(sp_col) {
  birds    <- unique(sp_col$bird_id)
  n_segs   <- SPERM_N - 1L
  seg_list <- vector("list", n_segs)
  for (i in seq_len(n_segs)) {
    parts <- lapply(birds, function(b) {
      bd <- sp_col[sp_col$bird_id == b & sp_col$tn > 1, ]
      bd <- bd[order(bd$timestamp), ]
      if (nrow(bd) <= i) return(NULL)
      data.frame(
        map_lon = c(bd$map_lon[i], bd$map_lon[i + 1], NA_real_),
        map_lat = c(bd$map_lat[i], bd$map_lat[i + 1], NA_real_)
      )
    })
    parts <- Filter(Negate(is.null), parts)
    if (length(parts)) seg_list[[i]] <- do.call(rbind, parts)
  }
  seg_list
}

Loading 9 CSVs...
Clean rows: 78464 | Birds: 173
Pre-computing interpolated tracks...


There were 50 or more warnings (use warnings() to see the first 50)


Ready. Interpolated rows: 49661


In [3]:
# ==============================================================================
# 2. UI
# ==============================================================================
ui <- page_navbar(
  title    = tags$span(bsicons::bs_icon("wind"), " Alpine Swift Migration Tracker"),
  theme    = bs_theme(version = 5, bootswatch = "flatly"),
  fillable = FALSE,

  tags$head(
    tags$style(HTML("
      .shiny-notification{position:fixed;top:20px;right:20px;bottom:auto;left:auto;z-index:9999;}
      .colony-row{display:flex;align-items:center;margin-bottom:4px;}
      .colony-dot{width:12px;height:12px;border-radius:50%;flex-shrink:0;margin-right:7px;}
      .colony-row label{font-size:.82rem;margin:0;cursor:pointer;}
      .value-box .value-box-value{font-size:1.35rem!important;}
    ")),
    tags$script(HTML("
      Shiny.addCustomMessageHandler('setColonyChecks', function(msg) {
        document.querySelectorAll('.colony-cb').forEach(function(cb){
          cb.checked = msg.val;
        });
        Shiny.setInputValue('colony_filter',
          msg.val ? msg.cols : [], {priority:'event'});
      });
    "))
  ),

  sidebar = sidebar(
    width = 280,
    h5(tags$b("Colonies"), class = "mt-1 mb-2"),
    uiOutput("colony_ui"),
    div(class = "d-flex gap-1 mb-2",
      actionButton("sel_all",  "All",  class = "btn-sm btn-outline-primary"),
      actionButton("sel_none", "None", class = "btn-sm btn-outline-secondary")
    ),
    hr(class = "my-2"),
    h5(tags$b("Individual")),
    selectInput("bird_sel", NULL, choices = c("All Birds", unique_birds), width = "100%"),
    hr(class = "my-2"),
    h5(tags$b("Animation")),
    radioButtons("play_speed", "Speed:",
                 choices = c("1×"=400, "2×"=200, "4×"=100), selected = 400, inline = TRUE),
    uiOutput("slider_ui"),
    hr(class = "my-2"),
    h5(tags$b("Visual Options")),
    checkboxInput("opt_sperm",  "Sperm trails (last 5 fixes)", value = TRUE),
    checkboxInput("opt_lines",  "Full flight paths",            value = FALSE),
    checkboxInput("opt_interp", "Smooth interpolated paths",   value = TRUE),
    hr(class = "my-2"),
    tags$p(class = "text-muted small mb-0",
      em("Lat: midpoint of geolocator uncertainty band"), tags$br(),
      em("Data: Movebank / Meier et al. 2020"))
  ),

  nav_panel(
    title = "Map & Analytics",
    layout_columns(
      col_widths = c(7, 5), gap = "1rem",

      card(
        full_screen = TRUE,
        card_header(tags$span(bsicons::bs_icon("map"), " Live Migration Map")),
        leafletOutput("map", width = "100%", height = "580px"),
        card_footer(class = "text-muted small",
          "Positions: midpoint of geolocator uncertainty interval + per-bird visual offset.")
      ),

      tagList(
        layout_columns(
          col_widths = c(4, 4, 4), gap = "0.5rem",
          value_box("GPS Fixes",    textOutput("stat_fixes"),
                    showcase = bsicons::bs_icon("crosshair"), theme = "primary",  height = "90px"),
          value_box("Active Birds", textOutput("stat_birds"),
                    showcase = bsicons::bs_icon("twitter"),   theme = "success",  height = "90px"),
          value_box("Avg Latitude", textOutput("stat_lat"),
                    showcase = bsicons::bs_icon("compass"),   theme = "info",     height = "90px")
        ),
        navset_card_underline(
          title = "Linked Analytics",
          nav_panel("Migration Profile",     plotOutput("plt_profile", height = "340px")),
          nav_panel("Latitude Distribution", plotOutput("plt_boxplot", height = "340px")),
          nav_panel("Colony Composition",    plotOutput("plt_pie",     height = "340px"))
        )
      )
    )
  )
)

Warning message:
Navigation containers expect a collection of `bslib::nav_panel()`/`shiny::tabPanel()`s and/or `bslib::nav_menu()`/`shiny::navbarMenu()`s. Consider using `header` or `footer` if you wish to place content above (or below) every panel's contents. 


In [4]:
# ==============================================================================
# 3. SERVER
# ==============================================================================
server <- function(input, output, session) {

  # ---- Colony UI ------------------------------------------------------------
  output$colony_ui <- renderUI({
    tagList(lapply(unique_colonies, function(col) {
      hex <- colony_colors[col]
      cid <- paste0("cb_", gsub("[^A-Za-z0-9]", "_", col))
      div(class = "colony-row",
        tags$input(class = "colony-cb form-check-input", type = "checkbox",
                   id = cid, value = col, checked = NA, style = "margin-top:0;",
                   onclick = "Shiny.setInputValue('colony_filter',
                     [...document.querySelectorAll('.colony-cb:checked')].map(e=>e.value),
                     {priority:'event'})"),
        div(class = "colony-dot", style = paste0("background:", hex, ";")),
        tags$label(`for` = cid, col)
      )
    }))
  })

  observeEvent(input$sel_all, {
    session$sendCustomMessage("setColonyChecks",
      list(val = TRUE, cols = as.list(unique_colonies)))
  })
  observeEvent(input$sel_none, {
    session$sendCustomMessage("setColonyChecks",
      list(val = FALSE, cols = as.list(unique_colonies)))
  })

  active_cols   <- reactive({
    cf <- input$colony_filter
    if (is.null(cf) || !length(cf)) unique_colonies else cf
  })
  active_cols_d <- debounce(active_cols, 250)

  # ---- Core data reactives --------------------------------------------------
  base_data <- reactive({
    d   <- swift_clean %>% filter(colony %in% active_cols_d())
    sel <- input$bird_sel
    if (!is.null(sel) && sel != "All Birds") d <- d %>% filter(bird_id == sel)
    d
  })

  interp_data <- reactive({
    d   <- swift_interp %>% filter(colony %in% active_cols_d())
    sel <- input$bird_sel
    if (!is.null(sel) && sel != "All Birds") d <- d %>% filter(bird_id == sel)
    d
  })

  path_data <- reactive({
    if (isTRUE(input$opt_interp)) interp_data() else base_data()
  })

  plot_base <- reactive({
    base_data() %>% arrange(bird_id, timestamp) %>%
      group_by(bird_id) %>%
      mutate(
        td  = as.numeric(difftime(timestamp, lag(timestamp), units = "days")),
        gap = as.integer(is.na(td) | td > 15),
        seg = cumsum(gap),
        grp = paste(bird_id, seg, sep = "_")
      ) %>% ungroup()
  })

  summer_gaps <- reactive({
    d <- base_data()
    if (nrow(d) < 2) return(NULL)
    vd <- sort(unique(d$timestamp))
    if (length(vd) < 2) return(NULL)
    tibble(s = vd[-length(vd)], e = vd[-1], df = as.numeric(diff(vd))) %>%
      filter(df > 15, month(s) %in% 5:8)
  })

  # ---- Slider ---------------------------------------------------------------
  output$slider_ui <- renderUI({
    req(base_data(), input$play_speed)
    d <- base_data()
    if (!nrow(d)) return(NULL)
    mn  <- min(d$timestamp, na.rm = TRUE)
    mx  <- max(d$timestamp, na.rm = TRUE)
    cur <- isolate(input$time_slider)
    if (is.null(cur) || cur < mn || cur > mx) cur <- mn
    sliderInput("time_slider", NULL, min = mn, max = mx, value = cur,
                timeFormat = "%Y-%m-%d", step = 1,
                animate = animationOptions(interval = as.numeric(input$play_speed), loop = FALSE))
  })

  observe({
    req(input$time_slider, summer_gaps())
    g <- summer_gaps()
    if (is.null(g) || !nrow(g)) return()
    hit <- g %>% filter(input$time_slider > s, input$time_slider < e)
    if (nrow(hit) > 0) {
      updateSliderInput(session, "time_slider", value = hit$e[1])
      showNotification(paste0("⏩ Skipping breeding gap → ", hit$e[1]),
                       type = "warning", duration = 4, id = "gap_msg")
    }
  })

  # ---- Map: static base (renders once, never again) -------------------------
  output$map <- renderLeaflet({
    leaflet(options = leafletOptions(minZoom = 2)) %>%
      addProviderTiles(providers$CartoDB.Positron) %>%
      setView(lng = 10, lat = 20, zoom = 3) %>%
      addLegend(position = "bottomright", pal = colony_palette,
                values = unique_colonies, title = "Colony", opacity = 0.9)
  })

  # ---- Map updates: NO FLICKER, NO LAG QUEUE --------------------------------
  #
  # Root causes fixed here:
  #
  # FLICKER: was caused by clearGroup() → redraw gap.
  #   Fix: use removeShape(layerId) to delete only specific named layers,
  #   then immediately add replacements. The old markers stay visible until
  #   the exact moment the new ones arrive — zero blank frames.
  #   For polylines (tracks/sperm) we still clear+redraw because they have
  #   no stable per-shape identity, but we batch them into one call per colony.
  #
  # LAG QUEUE: was caused by observe() firing on every slider tick, queuing
  #   renders faster than they complete. The throttle only rate-limited the
  #   reactive value, not the downstream computation which still accumulated.
  #   Fix: bindEvent() so the observer only fires when the slider actually
  #   *settles* for one full interval. Combined with req() guards to drop
  #   any stale invalidations immediately.

  # Compute the full map payload as one reactive — evaluated once per frame,
  # not once per layer. Shiny will discard intermediate values automatically.
  map_payload <- reactive({
    req(input$time_slider, base_data())
    ts <- input$time_slider

    # Current positions
    locs <- base_data() %>%
      filter(timestamp <= ts, timestamp >= ts - 7) %>%
      group_by(bird_id) %>% slice_tail(n = 1) %>% ungroup()

    # Sperm trail
    sp <- NULL
    if (isTRUE(isolate(input$opt_sperm))) {
      sp <- path_data() %>%
        filter(timestamp <= ts, !is.na(map_lat), !is.na(map_lon)) %>%
        group_by(bird_id) %>%
        slice_tail(n = SPERM_N) %>%
        mutate(ti = row_number(), tn = n()) %>%
        ungroup()
    }

    # Full tracks
    td <- NULL
    if (isTRUE(isolate(input$opt_lines))) {
      td <- path_data() %>% filter(timestamp <= ts, !is.na(map_lat))
    }

    list(ts = ts, locs = locs, sp = sp, td = td)
  }) %>%
    # KEY: debounce the entire payload — slider events that arrive faster than
    # 250ms are collapsed into one. Any pending computation is discarded.
    debounce(250)

  # Single observer, fires only when the debounced payload is ready.
  # Because map_payload is debounced, Shiny never queues more than one
  # pending render — old ones are thrown away automatically.
  observe({
    p <- map_payload()
    req(p)

    proxy <- leafletProxy("map")

    # ── Polylines: clear + batch redraw (unavoidable, but single call/colony) ──
    proxy <- proxy %>% clearGroup("track") %>% clearGroup("sperm")

    if (!is.null(p$td) && nrow(p$td) > 1) {
      for (col in unique(p$td$colony)) {
        seg <- p$td %>% filter(colony == col) %>% arrange(bird_id, timestamp)
        mat <- make_colony_lines(seg)
        if (!is.null(mat) && nrow(mat) > 1)
          proxy <- proxy %>%
            addPolylines(lng = mat$map_lon, lat = mat$map_lat,
                         color = colony_colors[col], weight = 1.5, opacity = 0.3,
                         group = "track")
      }
    }

    if (!is.null(p$sp) && nrow(p$sp) > 0) {
      for (col in unique(p$sp$colony)) {
        sp_col   <- p$sp %>% filter(colony == col)
        seg_list <- make_sperm_lines(sp_col)
        for (i in seq_along(seg_list)) {
          mat <- seg_list[[i]]
          if (is.null(mat) || nrow(mat) < 2) next
          frac <- i / SPERM_N
          proxy <- proxy %>%
            addPolylines(lng = mat$map_lon, lat = mat$map_lat,
                         color = colony_colors[col],
                         weight = 1.5 + 4.5 * frac,
                         opacity = 0.15 + 0.75 * frac,
                         group = "sperm")
        }
      }
    }

    # ── Circle markers: replace in-place with layerId — ZERO FLICKER ──────────
    # Each bird gets a stable layerId = its bird_id.
    # addCircleMarkers with an existing layerId *replaces* that marker atomically
    # on the JS side — the old marker is never removed first, so there is no
    # blank frame between old and new position.
    if (nrow(p$locs) > 0) {
      proxy <- proxy %>%
        addCircleMarkers(
          data        = p$locs,
          lng         = ~map_lon,
          lat         = ~map_lat,
          layerId     = ~bird_id,          # stable ID → atomic replacement
          radius      = 7,
          color       = "#222",
          weight      = 1.5,
          fillColor   = ~colony_palette(colony),
          fillOpacity = 0.95,
          popup       = ~paste0(
            "<b>Colony:</b> ", colony,    "<br>",
            "<b>Bird:</b> ",   bird_id,   "<br>",
            "<b>Date:</b> ",   timestamp, "<br>",
            "<b>Lat:</b> ",    round(lat_mid, 2), "°N<br>",
            "<b>Lon:</b> ",    round(lon_mid, 2), "°E"
          )
        )
      # Remove markers for birds that are no longer active this frame
      # (e.g. colony toggled off). Without this they'd linger forever.
      all_ids     <- unique(base_data()$bird_id)
      active_ids  <- p$locs$bird_id
      gone_ids    <- setdiff(all_ids, active_ids)
      if (length(gone_ids))
        proxy <- proxy %>% removeMarker(gone_ids)
    } else {
      # No active birds: clear all named markers
      proxy <- proxy %>% removeMarker(unique(base_data()$bird_id))
    }
  })

  # ---- Stats (driven by same debounced payload) -----------------------------
  locs_reactive <- reactive({
    p <- map_payload()
    req(p)
    p$locs
  })

  output$stat_fixes <- renderText(format(nrow(base_data()), big.mark = ","))
  output$stat_birds <- renderText(nrow(locs_reactive()))
  output$stat_lat   <- renderText({
    l <- locs_reactive()
    if (nrow(l)) paste0(round(mean(l$map_lat, na.rm = TRUE), 1), "° N") else "—"
  })

  # ---- Plots (also debounced via map_payload) --------------------------------
  output$plt_profile <- renderPlot({
    p <- map_payload()
    req(p, plot_base())
    ts  <- p$ts
    cur <- plot_base() %>% filter(timestamp >= ts - 60, timestamp <= ts)
    if (!nrow(cur)) return(NULL)
    tips <- cur %>% group_by(bird_id) %>% slice_tail(n = 1) %>% ungroup()
    ggplot(cur, aes(timestamp, plot_lat, color = colony, group = grp)) +
      geom_line(alpha = 0.5, linewidth = 0.65) +
      geom_point(data = tips, size = 2.2) +
      geom_vline(xintercept = as.numeric(ts),
                 color = "#d73027", linewidth = 0.9, linetype = "dashed") +
      scale_color_manual(values = colony_colors) +
      scale_x_date(limits = c(ts - 60, ts + 15), date_labels = "%b %Y") +
      scale_y_continuous(limits = c(-15, 55), breaks = seq(-10, 50, 10),
                         labels = function(x) paste0(x, "°N")) +
      theme_minimal(base_size = 11) +
      labs(x = NULL, y = "Latitude", color = "Colony",
           title = "Migration Profile (60-day window)") +
      theme(legend.position = "bottom", legend.text = element_text(size = 7.5),
            legend.key.size = unit(0.35, "cm"), panel.grid.minor = element_blank(),
            plot.title = element_text(size = 11, face = "bold"))
  }, res = 110)

  output$plt_boxplot <- renderPlot({
    d <- locs_reactive()
    if (!nrow(d)) return(NULL)
    ggplot(d, aes(reorder(colony, map_lat, FUN = median), map_lat, fill = colony)) +
      geom_boxplot(alpha = 0.8, outlier.shape = 21, outlier.size = 2,
                   width = 0.55, color = "#333") +
      scale_fill_manual(values = colony_colors) +
      scale_y_continuous(limits = c(-15, 55), breaks = seq(-10, 50, 10),
                         labels = function(x) paste0(x, "°N")) +
      theme_minimal(base_size = 11) +
      labs(x = NULL, y = "Latitude", title = "Current Latitude by Colony") +
      theme(legend.position = "none",
            axis.text.x = element_text(angle = 35, hjust = 1, size = 8.5),
            panel.grid.minor = element_blank(),
            plot.title = element_text(size = 11, face = "bold"))
  }, res = 110)

  output$plt_pie <- renderPlot({
    d <- locs_reactive()
    if (!nrow(d)) return(NULL)
    cnt <- d %>% group_by(colony) %>% summarise(n = n(), .groups = "drop") %>%
      mutate(pct = round(100 * n / sum(n)),
             lbl = paste0(colony, "\n", n, " (", pct, "%)"))
    ggplot(cnt, aes("", n, fill = colony)) +
      geom_bar(stat = "identity", width = 1, color = "white", linewidth = 0.7) +
      coord_polar("y", start = 0) +
      scale_fill_manual(values = colony_colors) +
      theme_void(base_size = 11) + labs(title = "Active Birds by Colony") +
      geom_text(aes(label = ifelse(pct >= 5, lbl, "")),
                position = position_stack(vjust = 0.5),
                color = "white", fontface = "bold", size = 3) +
      theme(legend.position = "none",
            plot.title = element_text(hjust = 0.5, face = "bold", size = 11))
  }, res = 110)
}




In [5]:
shinyApp(ui, server)

Warning in scale_x_date(limits = c(ts - 60, ts + 15), date_labels = "%b %Y") :
  A <numeric> value was passed to a Date scale.
ℹ The value was converted to a <Date> object.
Warning in scale_x_date(limits = c(ts - 60, ts + 15), date_labels = "%b %Y") :
  A <numeric> value was passed to a Date scale.
ℹ The value was converted to a <Date> object.
Warning in scale_x_date(limits = c(ts - 60, ts + 15), date_labels = "%b %Y") :
  A <numeric> value was passed to a Date scale.
ℹ The value was converted to a <Date> object.
Warning in scale_x_date(limits = c(ts - 60, ts + 15), date_labels = "%b %Y") :
  A <numeric> value was passed to a Date scale.
ℹ The value was converted to a <Date> object.
Warning in scale_x_date(limits = c(ts - 60, ts + 15), date_labels = "%b %Y") :
  A <numeric> value was passed to a Date scale.
ℹ The value was converted to a <Date> object.
Warning in scale_x_date(limits = c(ts - 60, ts + 15), date_labels = "%b %Y") :
  A <numeric> value was passed to a Date scale.
ℹ The va


Listening on http://127.0.0.1:6294
Input to asJSON(keep_vec_names=TRUE) is a named vector. In a future version of jsonlite, this option will not be supported, and named vectors will be translated into arrays instead of objects. If you want JSON object output, please use a named list instead. See ?toJSON.
Input to asJSON(keep_vec_names=TRUE) is a named vector. In a future version of jsonlite, this option will not be supported, and named vectors will be translated into arrays instead of objects. If you want JSON object output, please use a named list instead. See ?toJSON.
Input to asJSON(keep_vec_names=TRUE) is a named vector. In a future version of jsonlite, this option will not be supported, and named vectors will be translated into arrays instead of objects. If you want JSON object output, please use a named list instead. See ?toJSON.
Input to asJSON(keep_vec_names=TRUE) is a named vector. In a future version of jsonlite, this option will not be supported, and named vectors will be t

In [1]:
# Install these if you haven't already:
# install.packages(c("shiny", "bslib", "leaflet", "plotly", "dplyr", "lubridate", "RColorBrewer"))

library(shiny)
library(bslib)
library(leaflet)
library(plotly)
library(dplyr)
library(lubridate)
library(RColorBrewer)

# --- 1. DATA PREPARATION ---
# Helper function to load and format data safely
load_track_data <- function(filepath, pop_name) {
  if (file.exists(filepath)) {
    df <- read.csv(filepath)
    # Ensure standard column names and types based on Movebank format
    df <- df %>%
      mutate(
        population = pop_name,
        timestamp = as.POSIXct(timestamp, format="%Y-%m-%d %H:%M:%S", tz="UTC"),
        lat = location.lat,
        lon = location.long,
        bird_id = as.character(tag.local.identifier)
      ) %>%
      filter(!is.na(lat), !is.na(lon), !is.na(timestamp))
    return(df)
  }
  return(NULL)
}

# Load all populations (Update paths if your working directory changes)
tracks_list <- list(
  load_track_data("./data/Switzerland Baden - Long term study on migratory movement of Alpine swifts (Apus melba)-tracks.csv", "Switzerland Baden"),
  load_track_data("./data/Switzerland Biel - Long term study on migratory movement of Alpine swifts (Apus melba)-tracks.csv", "Switzerland Biel"),
  load_track_data("./data/Switzerland Lausanne - Long term study on migratory movement of Alpine swifts (Apus melba)-tracks.csv", "Switzerland Lausanne"),
  load_track_data("./data/Switzerland Lenzburg - Long term study on migratory movement of Alpine swifts (Apus melba)-tracks.csv", "Switzerland Lenzburg"),
  load_track_data("./data/Switzerland Luzern - Long term study on migratory movement of Alpine swifts (Apus melba)-tracks.csv", "Switzerland Luzern"),
  load_track_data("./data/Switzerland Solothurn - Long term study on migratory movement of Alpine swifts (Apus melba)-tracks.csv", "Switzerland Solothurn"),
  load_track_data("./data/Bulgaria Sofia - Long term study on migratory movement of Alpine swifts (Apus melba)-tracks.csv", "Bulgaria"),
  load_track_data("./data/Spain Tarragona - Long term study on migratory movement of Alpine swifts (Apus melba)-tracks.csv", "Spain"),
  load_track_data("./data/Turkey Pirasali - Long term study on migratory movement of Alpine swifts (Apus melba)-tracks.csv", "Turkey")
)

# Combine all loaded data
all_tracks <- bind_rows(tracks_list)

# If no data loaded, create dummy data to prevent app crash for demonstration
if (nrow(all_tracks) == 0) {
  warning("Data files not found. Using simulated data.")
  all_tracks <- data.frame(
    timestamp = seq(as.POSIXct("2014-08-01"), as.POSIXct("2015-05-31"), by="1 day"),
    lat = runif(304, 5, 47), lon = runif(304, -10, 30),
    population = "Simulated", bird_id = "Bird1"
  )
}

# Extract unique times (by day to keep animation smooth)
all_tracks <- all_tracks %>% mutate(date = as.Date(timestamp))
time_steps <- sort(unique(all_tracks$date))
pops <- unique(all_tracks$population)
birds <- unique(all_tracks$bird_id)

# Color palette
pop_colors <- colorRampPalette(brewer.pal(8, "Dark2"))(length(pops))
names(pop_colors) <- pops


# --- 2. USER INTERFACE (UI) using bslib to match your HTML ---
ui <- page_sidebar(
  title = "Alpine Swift Migration Tracker",
  theme = bs_theme(
    version = 5,
    bg = "#ffffff", fg = "#212529", primary = "#2c3e50",
    success = "#18bc9c", info = "#3498db"
  ),
  
  # SIDEBAR (Matches your HTML #sidebar)
  sidebar = sidebar(
    width = 300,
    bg = "#f8f9fa",
    
    tags$div(class="text-uppercase text-muted fw-bold", style="font-size:11px;", "Colonies"),
    checkboxGroupInput("colonies", NULL, choices = pops, selected = pops),
    actionButton("sel_all", "All", class="btn-sm btn-outline-primary"),
    actionButton("sel_none", "None", class="btn-sm btn-outline-primary"),
    hr(),
    
    tags$div(class="text-uppercase text-muted fw-bold", style="font-size:11px;", "Individual Bird"),
    selectInput("bird", NULL, choices = c("All Birds", birds)),
    hr(),
    
    tags$div(class="text-uppercase text-muted fw-bold", style="font-size:11px;", "Animation"),
    sliderInput("timeline", NULL, 
                min = min(time_steps), max = max(time_steps), 
                value = min(time_steps), step = 1,
                timeFormat = "%Y-%m-%d",
                animate = animationOptions(interval = 300, loop = FALSE)),
    hr(),
    
    tags$div(class="text-uppercase text-muted fw-bold", style="font-size:11px;", "Visual Options"),
    checkboxInput("trails", "Sperm trails (last 5 fixes)", value = TRUE),
    hr(),
    
    tags$small(class="text-muted", 
               "CRS: WGS84 (EPSG:4326)", br(), 
               "Data: Movebank / Meier & Liechti 2020")
  ),
  
  # MAIN AREA (Matches your HTML #main)
  
  # Value Boxes (Matches your HTML #vboxes)
  layout_columns(
    fill = FALSE,
    value_box(title = "GPS Fixes", value = textOutput("vb_fixes"), theme = "primary"),
    value_box(title = "Active Birds", value = textOutput("vb_birds"), theme = "success"),
    value_box(title = "Avg Latitude", value = textOutput("vb_lat"), theme = "info")
  ),
  
  # Map & Plots Row (Matches your HTML #mp-row)
  layout_columns(
    col_widths = c(7, 5),
    
    # Map Card
    card(
      card_header(class="fw-bold", "🗺 Live Migration Map"),
      leafletOutput("map", height = "100%"),
      card_footer(class="text-muted", style="font-size:11px;", 
                  "CRS: WGS84 · Colony anchors: Meier et al. 2020 (JAB)")
    ),
    
    # Analytics Tabs
    card(
      card_header(class="fw-bold", "Linked Analytics"),
      navset_underline(
        nav_panel("Migration Profile", plotlyOutput("plot_profile", height = "400px")),
        nav_panel("Lat Distribution", plotlyOutput("plot_dist", height = "400px")),
        nav_panel("Colony Comp", plotlyOutput("plot_comp", height = "400px"))
      )
    )
  )
)


# --- 3. SERVER LOGIC ---
server <- function(input, output, session) {
  
  # Select All / None buttons
  observeEvent(input$sel_all, { updateCheckboxGroupInput(session, "colonies", selected = pops) })
  observeEvent(input$sel_none, { updateCheckboxGroupInput(session, "colonies", selected = character(0)) })
  
  # Base reactive dataset filtered by colonies and bird selection (BUT NOT DATE YET)
  base_data <- reactive({
    req(input$colonies)
    df <- all_tracks %>% filter(population %in% input$colonies)
    if (input$bird != "All Birds") {
      df <- df %>% filter(bird_id == input$bird)
    }
    df
  })
  
  # Reactive data for the current date on the slider
  current_data <- reactive({
    req(input$timeline)
    base_data() %>% filter(date == input$timeline)
  })
  
  # Reactive data for trails (last 5 days)
  trail_data <- reactive({
    req(input$timeline, input$trails)
    start_date <- input$timeline - days(5)
    base_data() %>% 
      filter(date >= start_date & date <= input$timeline) %>%
      arrange(bird_id, date)
  })
  
  # --- Value Boxes ---
  output$vb_fixes <- renderText({ nrow(current_data()) })
  output$vb_birds <- renderText({ n_distinct(current_data()$bird_id) })
  output$vb_lat <- renderText({
    lats <- current_data()$lat
    if(length(lats) > 0) round(mean(lats, na.rm=TRUE), 2) else "—"
  })
  
  # --- Central Map ---
  output$map <- renderLeaflet({
    leaflet() %>%
      addProviderTiles(providers$CartoDB.Positron) %>%
      setView(lng = 15, lat = 30, zoom = 3) %>%
      addLegend("bottomright", colors = unname(pop_colors), 
                labels = names(pop_colors), title = "Colonies", opacity = 1)
  })
  
  # Update map markers and trails
  observe({
    pts <- current_data()
    
    proxy <- leafletProxy("map") %>% clearMarkers() %>% clearShapes()
    
    # Draw trails if enabled
    if(input$trails && nrow(trail_data()) > 0) {
      tr <- trail_data()
      for(b_id in unique(tr$bird_id)) {
        b_tr <- tr %>% filter(bird_id == b_id)
        if(nrow(b_tr) > 1) {
          proxy <- proxy %>%
            addPolylines(lng = b_tr$lon, lat = b_tr$lat, 
                         color = pop_colors[b_tr$population[1]],
                         weight = 2, opacity = 0.5)
        }
      }
    }
    
    # Draw current positions
    if(nrow(pts) > 0) {
      proxy %>%
        addCircleMarkers(data = pts, lng = ~lon, lat = ~lat,
                         radius = 5, color = "white", weight = 1,
                         fillColor = ~pop_colors[population], fillOpacity = 0.9,
                         popup = ~paste("Bird:", bird_id, "<br>Pop:", population))
    }
  })
  
  # --- Linked Plot 1: Migration Profile ---
  output$plot_profile <- renderPlotly({
    req(nrow(base_data()) > 0)
    # Show the journey up to the current date
    df <- base_data() %>% filter(date <= input$timeline)
    
    plot_ly(df, x = ~date, y = ~lat, color = ~population, colors = pop_colors, 
            type = 'scatter', mode = 'lines', line=list(width=1, opacity=0.5), 
            hoverinfo = "text", text = ~paste("Bird:", bird_id, "<br>Lat:", round(lat, 2))) %>%
      layout(xaxis = list(title = "", range = c(min(time_steps), max(time_steps))),
             yaxis = list(title = "Latitude"),
             showlegend = FALSE, margin = list(t=10, b=10))
  })
  
  # --- Linked Plot 2: Lat Distribution (Boxplot) ---
  output$plot_dist <- renderPlotly({
    df <- current_data()
    if(nrow(df) == 0) return(plotly_empty())
    
    plot_ly(df, y = ~lat, color = ~population, colors = pop_colors, type = "box") %>%
      layout(xaxis = list(title = ""), yaxis = list(title = "Latitude"),
             showlegend = FALSE, margin = list(t=10, b=10))
  })
  
  # --- Linked Plot 3: Colony Composition (Bar Chart instead of Pie) ---
  output$plot_comp <- renderPlotly({
    df <- current_data()
    if(nrow(df) == 0) return(plotly_empty())
    
    comp <- df %>% count(population)
    
    plot_ly(comp, x = ~population, y = ~n, color = ~population, colors = pop_colors, 
            type = 'bar') %>%
      layout(xaxis = list(title = ""), yaxis = list(title = "Active Birds"),
             showlegend = FALSE, margin = list(t=10, b=10))
  })
}

shinyApp(ui, server)


Attaching package: ‘bslib’

The following object is masked from ‘package:utils’:

    page



Warning message:
package ‘leaflet’ was built under R version 4.5.2 


Loading required package: ggplot2

Attaching package: ‘plotly’

The following object is masked from ‘package:ggplot2’:

    last_plot

The following object is masked from ‘package:stats’:

    filter

The following object is masked from ‘package:graphics’:

    layout



Warning message:
package ‘ggplot2’ was built under R version 4.5.3 



Attaching package: ‘dplyr’

The following objects are masked from ‘package:stats’:

    filter, lag

The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



Warning message:
package ‘dplyr’ was built under R version 4.5.3 



Attaching package: ‘lubridate’

The following objects are masked from ‘package:base’:

    date, intersect, setdiff, union




Listening on http://127.0.0.1:6550
Input to asJSON(keep_vec_names=TRUE) is a named vector. In a future version of jsonlite, this option will not be supported, and named vectors will be translated into arrays instead of objects. If you want JSON object output, please use a named list instead. See ?toJSON.
Input to asJSON(keep_vec_names=TRUE) is a named vector. In a future version of jsonlite, this option will not be supported, and named vectors will be translated into arrays instead of objects. If you want JSON object output, please use a named list instead. See ?toJSON.
Input to asJSON(keep_vec_names=TRUE) is a named vector. In a future version of jsonlite, this option will not be supported, and named vectors will be translated into arrays instead of objects. If you want JSON object output, please use a named list instead. See ?toJSON.
Input to asJSON(keep_vec_names=TRUE) is a named vector. In a future version of jsonlite, this option will not be supported, and named vectors will be t